This code will perform the following, load the dataset manually labeled, and then predict using a mlp model

# **Import Libraries**

In [7]:
print("Hello, World!")

Hello, World!


In [16]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from pathlib import Path
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# **1) EDA**

In [9]:
PATH = "kitchen_text_dataset.csv"
df = pd.read_csv(PATH)

In [10]:
df.head()

,tipo_comida,metodo_coccion,uso_aceite,hubo_derrame,cantidad_utensilios,cantidad_personas,nivel_suciedad_esperado
0,huevo,coccion_lenta,si,si,pocos,2,medio
1,agua,salteado,si,no,muchos,2,medio
2,huevo,hervido,si,no,pocos,3_o_mas,bajo
3,pollo,recalentado,si,si,muchos,3_o_mas,alto
4,pescado,hervido,si,no,muchos,3_o_mas,alto


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   tipo_comida              498 non-null    str  
 1   metodo_coccion           498 non-null    str  
 2   uso_aceite               498 non-null    str  
 3   hubo_derrame             498 non-null    str  
 4   cantidad_utensilios      498 non-null    str  
 5   cantidad_personas        498 non-null    str  
 6   nivel_suciedad_esperado  498 non-null    str  
dtypes: str(7)
memory usage: 27.4 KB


In [ ]:
def build_and_export_text_dataset(
    df,
    output_dir="data",
    test_size=0.15,
    val_size=0.15,
    random_state=42
):
    """
    Preprocesses the kitchen contextual dataset and exports:

    DATA:
    - train.csv
    - val.csv
    - test.csv

    PREPROCESSING OBJECTS:
    - preprocessor.joblib
    - label_encoder.joblib

    Operations:
    - maps yes/no -> 1/0
    - one-hot encodes categorical columns
    - label encodes target

    Returns:
        X_train
        X_val
        X_test
        y_train
        y_val
        y_test
        preprocessor
        label_encoder
    """

    df = df.copy()
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # BINARY MAPPING
    binary_mapping = {
        "si": 1,
        "no": 0
    }

    binary_columns = [
        "uso_aceite",
        "hubo_derrame"
    ]

    for col in binary_columns:
        df[col] = df[col].map(binary_mapping)

    # FEATURES / TARGET
    X = df.drop(columns=["nivel_suciedad_esperado"])
    y = df["nivel_suciedad_esperado"]

    # LABEL ENCODER
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # CATEGORICAL COLUMNS
    categorical_columns = [
        "tipo_comida",
        "metodo_coccion",
        "cantidad_utensilios",
        "cantidad_personas"
    ]

    # PREPROCESSOR
    preprocessor = ColumnTransformer(
        transformers=[

            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_columns
            )

        ],
        remainder="passthrough"
    )

    # TRANSFORM FEATURES
    X_processed = preprocessor.fit_transform(X)

    # FEATURE NAMES
    feature_names = (
        preprocessor
        .get_feature_names_out()
    )

    # CREATE PROCESSED DATAFRAME
    processed_df = pd.DataFrame(
        X_processed,
        columns=feature_names
    )

    processed_df["target"] = y_encoded

    # TRAIN / TEST SPLIT
    train_df, test_df = train_test_split(
        processed_df,
        test_size=test_size,
        stratify=processed_df["target"],
        random_state=random_state
    )

    # TRAIN / VAL SPLIT
    adjusted_val_size = val_size / (
        1 - test_size
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=adjusted_val_size,
        stratify=train_df["target"],
        random_state=random_state
    )

    # EXPORT DATASETS
    train_df.to_csv(
        output_dir / "train.csv",
        index=False
    )

    val_df.to_csv(
        output_dir / "val.csv",
        index=False
    )

    test_df.to_csv(
        output_dir / "test.csv",
        index=False
    )

    # SAVE PREPROCESSING OBJECTS
    joblib.dump(
        preprocessor,
        output_dir / "preprocessor.joblib"
    )

    joblib.dump(
        label_encoder,
        output_dir / "label_encoder.joblib"
    )

    # SPLIT FEATURES / LABELS
    X_train = train_df.drop(
        columns=["target"]
    ).values
    y_train = train_df["target"].values
    X_val = val_df.drop(
        columns=["target"]
    ).values
    y_val = val_df["target"].values
    X_test = test_df.drop(
        columns=["target"]
    ).values
    y_test = test_df["target"].values

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        preprocessor,
        label_encoder
    )

In [13]:
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    preprocessor,
    label_encoder

) = build_and_export_text_dataset(
    df=df,
    output_dir="data",
    test_size=0.15,
    val_size=0.15,
    random_state=42
)

# **MLP**

In [15]:
train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

X_train = train_df.drop(columns=["target"]).values
y_train = train_df["target"].values

X_val = val_df.drop(columns=["target"]).values
y_val = val_df["target"].values

X_test = test_df.drop(columns=["target"]).values
y_test = test_df["target"].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (348, 30)
y_train shape: (348,)
X_val shape: (75, 30)
y_val shape: (75,)
X_test shape: (75, 30)
y_test shape: (75,)


In [ ]:
import json

def build_and_train_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    num_classes=3,
    epochs=50,
    batch_size=16,
    learning_rate=1e-3
):
    """
    Builds and trains a simple MLP
    for contextual dirtiness classification.
    """

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(X_train.shape[1],)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            32,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax"
        )
    ])

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="sparse_categorical_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    model.summary()

    history = model.fit(

        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=epochs,
        batch_size=batch_size
    )

    return model, history


# ====================================================
# EVALUATION FUNCTION
# ====================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    label_encoder,
    model_name="mlp_model",
    results_dir="results"
):
    """
    Evaluates MLP model and saves:

    - metrics json
    - confusion matrix csv
    - trained model
    """

    # ------------------------------------------------
    # CREATE RESULTS DIRECTORY
    # ------------------------------------------------

    results_dir = Path(results_dir)

    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------
    # INFERENCE
    # ------------------------------------------------

    start_time = time.time()

    y_probs = model.predict(X_test)

    end_time = time.time()

    inference_time = (
        end_time - start_time
    )

    # ------------------------------------------------
    # PREDICTIONS
    # ------------------------------------------------

    y_pred = np.argmax(
        y_probs,
        axis=1
    )

    # ------------------------------------------------
    # METRICS
    # ------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    # ------------------------------------------------
    # METRICS DICTIONARY
    # ------------------------------------------------

    metrics = {

        "model_name": model_name,

        "accuracy": float(accuracy),

        "precision": float(precision),

        "recall": float(recall),

        "f1_score": float(f1),

        "inference_time_seconds": float(
            inference_time
        ),

        "num_test_samples": int(
            len(y_test)
        ),

        "classification_report": report,

        "confusion_matrix": cm.tolist()
    }

    # ------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------

    print("\n========== MLP METRICS ==========\n")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print(
        f"\nInference Time: "
        f"{inference_time:.4f} seconds"
    )

    print(
        "\n========== CLASSIFICATION REPORT ==========\n"
    )

    print(

        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_
        )
    )

    # ------------------------------------------------
    # SAVE MODEL
    # ------------------------------------------------

    model.save(
        results_dir / f"{model_name}.keras"
    )

    # ------------------------------------------------
    # SAVE METRICS JSON
    # ------------------------------------------------

    metrics_path = (
        results_dir /
        f"{model_name}_metrics.json"
    )

    with open(
        metrics_path,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    # ------------------------------------------------
    # SAVE CONFUSION MATRIX CSV
    # ------------------------------------------------

    cm_df = pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    )

    cm_df.to_csv(
        results_dir /
        f"{model_name}_confusion_matrix.csv"
    )

    # ------------------------------------------------
    # SAVE PREPROCESSING OBJECTS
    # ------------------------------------------------

    joblib.dump(
        label_encoder,
        results_dir /
        "label_encoder.joblib"
    )

    print(
        f"\nResults saved in: {results_dir}"
    )

    return metrics


# ====================================================
# TRAIN MODEL
# ====================================================

model, history = build_and_train_mlp(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    num_classes=3,
    epochs=50,
    batch_size=16,
    learning_rate=1e-3
)


# ====================================================
# EVALUATE MODEL
# ====================================================

metrics = evaluate_model(
    model=model,
    X_test=X_test,
    y_test=y_test,
    label_encoder=label_encoder,
    model_name="kitchen_context_mlp",
    results_dir="results"
)